Imports

In [1]:
from qiskit_metal import designs
from qiskit_metal import MetalGUI, Dict, open_docs
from qiskit_metal.qlibrary.tlines.meandered import RouteMeander
from qiskit_metal.qlibrary.tlines.pathfinder import RoutePathfinder
from qiskit_metal.qlibrary.terminations.launchpad_wb_driven import LaunchpadWirebondDriven
from qiskit_metal.qlibrary.terminations.open_to_ground import OpenToGround
from qiskit_metal.qlibrary.couplers.coupled_line_tee import CoupledLineTee
import pyEPR as epr

#libraries for proper port definition
import numpy as np
from qiskit_metal.renderers.renderer_ansys.hfss_renderer import QHFSSRenderer
from qiskit_metal.draw.utility import to_vec3D
from pyEPR.ansys import parse_units

def create_ports_g2s(self, port_list):
    for qcomp, pin, impedance in port_list:
        port_name = f'Port_{qcomp}_{pin}'
        pdict = self.design.components[qcomp].pins[pin]
        midpt, gap_size = pdict['middle'], pdict['gap']
        norm_vec, width = pdict['normal'], parse_units(pdict['width'])

        endpoints = parse_units([midpt, midpt + gap_size * norm_vec])
        endpoints_3d = to_vec3D(endpoints, 0)
        x0, y0 = endpoints_3d[0][:2]
        x1, y1 = endpoints_3d[1][:2]
        if abs(y1 - y0) > abs(x1 - x0):
            x_min, x_max = x0 - width / 2, x0 + width / 2
            y_min, y_max = min(y0, y1), max(y0, y1)
        else:
            x_min, x_max = min(x0, x1), max(x0, x1)
            y_min, y_max = y0 - width / 2, y0 + width / 2

        poly_ansys = self.modeler.draw_rect_corner(
            [x_min, y_min, 0], x_max - x_min, y_max - y_min, 0,
            **dict(transparency=0.0))

        start = list(endpoints_3d[0])   # ground side of gap
        end   = list(endpoints_3d[1])   # signal conductor edge

        if self.pinfo.design.solution_type != 'Eigenmode':
            self.modeler._make_lumped_port(
                start, end, ["Objects:=", [poly_ansys]],
                z0=str(impedance) + 'ohm',
                name=f'LumpPort_{qcomp}_{pin}')
            self.modeler.rename_obj(poly_ansys, port_name)
        else:
            self.modeler._make_lumped_rlc(
                str(impedance) + 'ohm', 0, 0, start, end,
                ["Objects:=", [poly_ansys]],
                name=f'RLCBoundary_{qcomp}_{pin}')
            self.modeler.rename_obj(poly_ansys, port_name)
            self.assign_port_mesh.append(port_name)

QHFSSRenderer.create_ports = create_ports_g2s

# Closes all previous GUI instances when running the program again
for _g in ['gui1','gui2','gui3','gui4','gui5','gui6','gui7','gui8','gui9','gui10']:
    try:
        _gui = globals()[_g]
        _gui.main_window.force_close = True
        _gui.main_window.close()
    except (KeyError, NameError, AttributeError):
        pass


Universal Parameters

In [2]:
main_x = '5mm'
main_y = '5mm'
main_z = '-380um'
radbound_top='2mm'
radbound_bottom='380um'

lp_width = '180um'
lp_gap = '140um'

tl_width = '15.5um'
tl_gap = '9um'

rr_width = tl_width
rr_gap = tl_gap
rr_coupling_gap = '15um' 
rr_coupling_length = '850um'
rr_termination = rr_gap

## Resonator 1: 1mm

In [3]:
#design and GUI intialization
R1 = designs.DesignPlanar()
R1.overwrite_enabled = True
gui1 = MetalGUI(R1)

#chip & sample holder dimensions
R1.chips.main.size.size_x = main_x
R1.chips.main.size.size_y = main_y
R1.chips.main.size.size_z = main_z
R1.chips.main.size.sample_holder_top = radbound_top
R1.chips.main.size.sample_holder_bottom = radbound_bottom

#launch pad definition
R1_LP1 = LaunchpadWirebondDriven(R1,name='LP1',
                       options={'orientation': '0', 'pos_x': '-2.1mm', 'pos_y': '0mm','pad_width':lp_width ,'pad_gap':lp_gap ,'pad_height':'180um', 'trace_width': tl_width , 'trace_gap': tl_gap},
                       component_template={'falseparam1': {'falseparam2': 'false-param', 'falseparam3': 'false-param'}},
                                        )

R1_LP2 = LaunchpadWirebondDriven(R1,name='LP2',
                       options={'orientation': '180', 'pos_x': '2.1mm', 'pos_y': '0mm','pad_width':lp_width ,'pad_gap':lp_gap, 'pad_height':'180um', 'trace_width':tl_width , 'trace_gap': tl_gap},
                       component_template={'falseparam1': {'falseparam2': 'false-param', 'falseparam3': 'false-param'}},
                                        )

#resonator definition
R1_OTG = OpenToGround(R1,name='OTG'
                 ,options={'pos_x':'-0.425mm','pos_y':'-0.2985mm','orientation':'-90','width':rr_width, 'gap':rr_gap, 'termination_gap': rr_termination })


R1_CLT = CoupledLineTee(R1,'CLT', options=dict(pos_x = '0mm', pos_y = '0mm',
                                                        prime_width=tl_width,
                                                        prime_gap=tl_gap,
                                                        second_width=rr_width,
                                                        second_gap=rr_gap,
                                                        fillet='50um',
                                                        mirror=True,
                                                        orientation = '0',
                                                        coupling_space = rr_coupling_gap,                                                         
                                                        coupling_length = rr_coupling_length,
                                                        open_termination = False))

RR1 = RouteMeander(R1, 'RR1',  Dict(
        trace_width =tl_width,
        trace_gap =tl_gap,
        total_length='150um',
        hfss_wire_bonds = True,
        fillet='50 um',
        lead = dict(start_straight='100um',end_straight='10um'),
        meander= dict(spacing='150um',asymmetry='200um'),
        pin_inputs=Dict(
            start_pin=Dict(component='CLT', pin='second_end'),
            end_pin=Dict(component='OTG', pin='open')), ))

#Tx line segments

seg1_1 = RoutePathfinder(R1, 'seg1', options = dict(chip='main', 
                                                      trace_width =tl_width,
                                                      trace_gap =tl_gap,
                                                      fillet='100um',
                                                      hfss_wire_bonds = True,
                                                    lead=dict(start_straight = '0mm',
                                                        end_straight='0mm'),
                                                        pin_inputs=dict(
                                                        start_pin=dict(
                                                                component='LP1',
                                                                pin='tie'),
                                                       end_pin=dict(
                                                        component='CLT',
                                                        pin='prime_start')
                                                 )))


seg2_1 = RoutePathfinder(R1, 'seg2', options = dict(chip='main', 
                                                      trace_width =tl_width,
                                                      trace_gap =tl_gap,
                                                      fillet='100um',
                                                      hfss_wire_bonds = True,
                                                    lead=dict(start_straight = '0mm',
                                                        end_straight='0mm'),
                                                        pin_inputs=dict(
                                                        start_pin=dict(
                                                                component='CLT',
                                                                pin='prime_end'),
                                                       end_pin=dict(
                                                        component='LP2',
                                                        pin='tie')
                                                 )))


gui1.rebuild()
gui1.autoscale()


04:14PM 40s INFO [_start_renderers]: Renderer=gmsh skipped: runtime dependency not installed (renderer_gmsh requires gmsh. Install with: pip install 'quantum-metal[mesh]' (or the legacy alias 'quantum-metal[fem]')).
04:14PM 43s INFO [connect_meandered]: Zero meanders for RR1
04:14PM 43s INFO [connect_meandered]: Zero meanders for RR1


04:15PM 47s CRITICAL [_qt_message_handler]: CRITICAL: QEventDispatcherWin32::wakeUp: Failed to post a message (Not enough quota is available to process this command.) (No context available from Qt)
Python Traceback (most recent call last):
  File "c:\Users\labuser\miniconda3\envs\qm\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\labuser\miniconda3\envs\qm\Lib\site-packages\traitlets\config\application.py", line 1082, in launch_instance
    app.start()
  File "c:\Users\labuser\miniconda3\envs\qm\Lib\site-packages\ipykernel\kernelapp.py", line 758, in start
    self.io_loop.start()
  File "c:\Users\labuser\miniconda3\envs\qm\Lib\site-packages\tornado\platform\asyncio.py", line 211, in start
    self.asyncio_loop.run_forever()
  File "c:\Users\labuser\miniconda3\envs\qm\Lib\asyncio\base_events.py", line 608, in run_forever
    self._run_once()
  File "c:\Users\labuser\miniconda3\envs\qm\Lib\asyncio\base_events.py", line 1936, 

## Resonator 2: 2mm

In [4]:
#design and GUI intialization
R2 = designs.DesignPlanar()
R2.overwrite_enabled = True
gui2 = MetalGUI(R2)

#chip & sample holder dimensions
R2.chips.main.size.size_x = main_x
R2.chips.main.size.size_y = main_y
R2.chips.main.size.size_z = main_z
R2.chips.main.size.sample_holder_top = radbound_top
R2.chips.main.size.sample_holder_bottom = radbound_bottom

#launch pad definition
R2_LP1 = LaunchpadWirebondDriven(R2,name='LP1',
                       options={'orientation': '0', 'pos_x': '-2.1mm', 'pos_y': '0mm','pad_width':lp_width ,'pad_gap':lp_gap ,'pad_height':'180um', 'trace_width': tl_width , 'trace_gap': tl_gap},
                       component_template={'falseparam1': {'falseparam2': 'false-param', 'falseparam3': 'false-param'}},
                                        )

R2_LP2 = LaunchpadWirebondDriven(R2,name='LP2',
                       options={'orientation': '180', 'pos_x': '2.1mm', 'pos_y': '0mm','pad_width':lp_width ,'pad_gap':lp_gap, 'pad_height':'180um', 'trace_width':tl_width , 'trace_gap': tl_gap},
                       component_template={'falseparam1': {'falseparam2': 'false-param', 'falseparam3': 'false-param'}},
                                        )

#resonator definition
R2_OTG = OpenToGround(R2,name='OTG'
                 ,options={'pos_x':'-0.425mm','pos_y':'-0.8485mm','orientation':'-90','width':rr_width, 'gap':rr_gap, 'termination_gap': rr_termination })


R2_CLT = CoupledLineTee(R2,'CLT', options=dict(pos_x = '0mm', pos_y = '0mm',
                                                        prime_width=tl_width,
                                                        prime_gap=tl_gap,
                                                        second_width=rr_width,
                                                        second_gap=rr_gap,
                                                        fillet='50um',
                                                        mirror=True,
                                                        orientation = '0',
                                                        coupling_space = rr_coupling_gap,                                                         
                                                        coupling_length = rr_coupling_length,
                                                        open_termination = False))

RR2 = RouteMeander(R2, 'RR2',  Dict(
        trace_width =tl_width,
        trace_gap =tl_gap,
        total_length='1150um',
        hfss_wire_bonds = True,
        fillet='50 um',
        lead = dict(start_straight='100um',end_straight='200um'),
        meander= dict(spacing='150um',asymmetry='200um'),
        pin_inputs=Dict(
            start_pin=Dict(component='CLT', pin='second_end'),
            end_pin=Dict(component='OTG', pin='open')), ))

#Tx line segments

seg1_2 = RoutePathfinder(R2, 'seg1', options = dict(chip='main', 
                                                      trace_width =tl_width,
                                                      trace_gap =tl_gap,
                                                      fillet='100um',
                                                      hfss_wire_bonds = True,
                                                    lead=dict(start_straight = '0mm',
                                                        end_straight='0mm'),
                                                        pin_inputs=dict(
                                                        start_pin=dict(
                                                                component='LP1',
                                                                pin='tie'),
                                                       end_pin=dict(
                                                        component='CLT',
                                                        pin='prime_start')
                                                 )))


seg2_2 = RoutePathfinder(R2, 'seg2', options = dict(chip='main', 
                                                      trace_width =tl_width,
                                                      trace_gap =tl_gap,
                                                      fillet='100um',
                                                      hfss_wire_bonds = True,
                                                    lead=dict(start_straight = '0mm',
                                                        end_straight='0mm'),
                                                        pin_inputs=dict(
                                                        start_pin=dict(
                                                                component='CLT',
                                                                pin='prime_end'),
                                                       end_pin=dict(
                                                        component='LP2',
                                                        pin='tie')
                                                 )))


gui2.rebuild()
gui2.autoscale()

04:14PM 44s INFO [_start_renderers]: Renderer=gmsh skipped: runtime dependency not installed (renderer_gmsh requires gmsh. Install with: pip install 'quantum-metal[mesh]' (or the legacy alias 'quantum-metal[fem]')).


## Resonator 3: 3mm

In [5]:
#design and GUI intialization
R3 = designs.DesignPlanar()
R3.overwrite_enabled = True
gui3 = MetalGUI(R3)

#chip & sample holder dimensions
R3.chips.main.size.size_x = main_x
R3.chips.main.size.size_y = main_y
R3.chips.main.size.size_z = main_z
R3.chips.main.size.sample_holder_top = radbound_top
R3.chips.main.size.sample_holder_bottom = radbound_bottom

#launch pad definition
R3_LP1 = LaunchpadWirebondDriven(R3,name='LP1',
                       options={'orientation': '0', 'pos_x': '-2.1mm', 'pos_y': '0mm','pad_width':lp_width ,'pad_gap':lp_gap ,'pad_height':'180um', 'trace_width': tl_width , 'trace_gap': tl_gap},
                       component_template={'falseparam1': {'falseparam2': 'false-param', 'falseparam3': 'false-param'}},
                                        )

R3_LP2 = LaunchpadWirebondDriven(R3,name='LP2',
                       options={'orientation': '180', 'pos_x': '2.1mm', 'pos_y': '0mm','pad_width':lp_width ,'pad_gap':lp_gap, 'pad_height':'180um', 'trace_width':tl_width , 'trace_gap': tl_gap},
                       component_template={'falseparam1': {'falseparam2': 'false-param', 'falseparam3': 'false-param'}},
                                        )

#resonator definition
R3_OTG = OpenToGround(R3,name='OTG'
                 ,options={'pos_x':'-0.425mm','pos_y':'-1.0mm','orientation':'-90','width':rr_width, 'gap':rr_gap, 'termination_gap': rr_termination })


R3_CLT = CoupledLineTee(R3,'CLT', options=dict(pos_x = '0mm', pos_y = '0mm',
                                                        prime_width=tl_width,
                                                        prime_gap=tl_gap,
                                                        second_width=rr_width,
                                                        second_gap=rr_gap,
                                                        fillet='50um',
                                                        mirror=True,
                                                        orientation = '0',
                                                        coupling_space = rr_coupling_gap,                                                         
                                                        coupling_length = rr_coupling_length,
                                                        open_termination = False))

RR3 = RouteMeander(R3, 'RR3',  Dict(
        trace_width =tl_width,
        trace_gap =tl_gap,
        total_length='2150um',
        hfss_wire_bonds = True,
        fillet='50 um',
        lead = dict(start_straight='100um',end_straight='10um'),
        meander= dict(spacing='150um',asymmetry='200um'),
        pin_inputs=Dict(
            start_pin=Dict(component='CLT', pin='second_end'),
            end_pin=Dict(component='OTG', pin='open')), ))

#Tx line segments

seg1_3 = RoutePathfinder(R3, 'seg1', options = dict(chip='main', 
                                                      trace_width =tl_width,
                                                      trace_gap =tl_gap,
                                                      fillet='100um',
                                                      hfss_wire_bonds = True,
                                                    lead=dict(start_straight = '0mm',
                                                        end_straight='0mm'),
                                                        pin_inputs=dict(
                                                        start_pin=dict(
                                                                component='LP1',
                                                                pin='tie'),
                                                       end_pin=dict(
                                                        component='CLT',
                                                        pin='prime_start')
                                                 )))


seg2_3 = RoutePathfinder(R3, 'seg2', options = dict(chip='main', 
                                                      trace_width =tl_width,
                                                      trace_gap =tl_gap,
                                                      fillet='100um',
                                                      hfss_wire_bonds = True,
                                                    lead=dict(start_straight = '0mm',
                                                        end_straight='0mm'),
                                                        pin_inputs=dict(
                                                        start_pin=dict(
                                                                component='CLT',
                                                                pin='prime_end'),
                                                       end_pin=dict(
                                                        component='LP2',
                                                        pin='tie')
                                                 )))


gui3.rebuild()
gui3.autoscale()


04:14PM 45s INFO [_start_renderers]: Renderer=gmsh skipped: runtime dependency not installed (renderer_gmsh requires gmsh. Install with: pip install 'quantum-metal[mesh]' (or the legacy alias 'quantum-metal[fem]')).


## Resonator 4: 4mm

In [6]:
#design and GUI intialization
R4 = designs.DesignPlanar()
R4.overwrite_enabled = True
gui4 = MetalGUI(R4)

#chip & sample holder dimensions
R4.chips.main.size.size_x = main_x
R4.chips.main.size.size_y = main_y
R4.chips.main.size.size_z = main_z
R4.chips.main.size.sample_holder_top = radbound_top
R4.chips.main.size.sample_holder_bottom = radbound_bottom

#launch pad definition
R4_LP1 = LaunchpadWirebondDriven(R4,name='LP1',
                       options={'orientation': '0', 'pos_x': '-2.1mm', 'pos_y': '0mm','pad_width':lp_width ,'pad_gap':lp_gap ,'pad_height':'180um', 'trace_width': tl_width , 'trace_gap': tl_gap},
                       component_template={'falseparam1': {'falseparam2': 'false-param', 'falseparam3': 'false-param'}},
                                        )

R4_LP2 = LaunchpadWirebondDriven(R4,name='LP2',
                       options={'orientation': '180', 'pos_x': '2.1mm', 'pos_y': '0mm','pad_width':lp_width ,'pad_gap':lp_gap, 'pad_height':'180um', 'trace_width':tl_width , 'trace_gap': tl_gap},
                       component_template={'falseparam1': {'falseparam2': 'false-param', 'falseparam3': 'false-param'}},
                                        )

#resonator definition
R4_OTG = OpenToGround(R4,name='OTG'
                 ,options={'pos_x':'-0.425mm','pos_y':'-1.0mm','orientation':'-90','width':rr_width, 'gap':rr_gap, 'termination_gap': rr_termination })


R4_CLT = CoupledLineTee(R4,'CLT', options=dict(pos_x = '0mm', pos_y = '0mm',
                                                        prime_width=tl_width,
                                                        prime_gap=tl_gap,
                                                        second_width=rr_width,
                                                        second_gap=rr_gap,
                                                        fillet='50um',
                                                        mirror=True,
                                                        orientation = '0',
                                                        coupling_space = rr_coupling_gap,                                                         
                                                        coupling_length = rr_coupling_length,
                                                        open_termination = False))

RR4 = RouteMeander(R4, 'RR4',  Dict(
        trace_width =tl_width,
        trace_gap =tl_gap,
        total_length='3150um',
        hfss_wire_bonds = True,
        fillet='50 um',
        lead = dict(start_straight='100um',end_straight='100um'),
        meander= dict(spacing='150um',asymmetry='200um'),
        pin_inputs=Dict(
            start_pin=Dict(component='CLT', pin='second_end'),
            end_pin=Dict(component='OTG', pin='open')), ))

#Tx line segments

seg1_4 = RoutePathfinder(R4, 'seg1', options = dict(chip='main', 
                                                      trace_width =tl_width,
                                                      trace_gap =tl_gap,
                                                      fillet='100um',
                                                      hfss_wire_bonds = True,
                                                    lead=dict(start_straight = '0mm',
                                                        end_straight='0mm'),
                                                        pin_inputs=dict(
                                                        start_pin=dict(
                                                                component='LP1',
                                                                pin='tie'),
                                                       end_pin=dict(
                                                        component='CLT',
                                                        pin='prime_start')
                                                 )))


seg2_4 = RoutePathfinder(R4, 'seg2', options = dict(chip='main', 
                                                      trace_width =tl_width,
                                                      trace_gap =tl_gap,
                                                      fillet='100um',
                                                      hfss_wire_bonds = True,
                                                    lead=dict(start_straight = '0mm',
                                                        end_straight='0mm'),
                                                        pin_inputs=dict(
                                                        start_pin=dict(
                                                                component='CLT',
                                                                pin='prime_end'),
                                                       end_pin=dict(
                                                        component='LP2',
                                                        pin='tie')
                                                 )))


gui4.rebuild()
gui4.autoscale()


04:14PM 45s INFO [_start_renderers]: Renderer=gmsh skipped: runtime dependency not installed (renderer_gmsh requires gmsh. Install with: pip install 'quantum-metal[mesh]' (or the legacy alias 'quantum-metal[fem]')).


## Resonator 5: 5mm


In [7]:
#design and GUI intialization
R5 = designs.DesignPlanar()
R5.overwrite_enabled = True
gui5 = MetalGUI(R5)

#chip & sample holder dimensions
R5.chips.main.size.size_x = main_x
R5.chips.main.size.size_y = main_y
R5.chips.main.size.size_z = main_z
R5.chips.main.size.sample_holder_top = radbound_top
R5.chips.main.size.sample_holder_bottom = radbound_bottom

#launch pad definition
R5_LP1 = LaunchpadWirebondDriven(R5,name='LP1',
                       options={'orientation': '0', 'pos_x': '-2.1mm', 'pos_y': '0mm','pad_width':lp_width ,'pad_gap':lp_gap ,'pad_height':'180um', 'trace_width': tl_width , 'trace_gap': tl_gap},
                       component_template={'falseparam1': {'falseparam2': 'false-param', 'falseparam3': 'false-param'}},
                                        )

R5_LP2 = LaunchpadWirebondDriven(R5,name='LP2',
                       options={'orientation': '180', 'pos_x': '2.1mm', 'pos_y': '0mm','pad_width':lp_width ,'pad_gap':lp_gap, 'pad_height':'180um', 'trace_width':tl_width , 'trace_gap': tl_gap},
                       component_template={'falseparam1': {'falseparam2': 'false-param', 'falseparam3': 'false-param'}},
                                        )

#resonator definition
R5_OTG = OpenToGround(R5,name='OTG'
                 ,options={'pos_x':'-0.425mm','pos_y':'-1.0mm','orientation':'-90','width':rr_width, 'gap':rr_gap, 'termination_gap': rr_termination })


R5_CLT = CoupledLineTee(R5,'CLT', options=dict(pos_x = '0mm', pos_y = '0mm',
                                                        prime_width=tl_width,
                                                        prime_gap=tl_gap,
                                                        second_width=rr_width,
                                                        second_gap=rr_gap,
                                                        fillet='50um',
                                                        mirror=True,
                                                        orientation = '0',
                                                        coupling_space = rr_coupling_gap,                                                         
                                                        coupling_length = rr_coupling_length,
                                                        open_termination = False))

RR5 = RouteMeander(R5, 'RR5',  Dict(
        trace_width =tl_width,
        trace_gap =tl_gap,
        total_length='4150um',
        hfss_wire_bonds = True,
        fillet='50 um',
        lead = dict(start_straight='100um',end_straight='10um'),
        meander= dict(spacing='150um',asymmetry='200um'),
        pin_inputs=Dict(
            start_pin=Dict(component='CLT', pin='second_end'),
            end_pin=Dict(component='OTG', pin='open')), ))

#Tx line segments

seg1_5 = RoutePathfinder(R5, 'seg1', options = dict(chip='main', 
                                                      trace_width =tl_width,
                                                      trace_gap =tl_gap,
                                                      fillet='100um',
                                                      hfss_wire_bonds = True,
                                                    lead=dict(start_straight = '0mm',
                                                        end_straight='0mm'),
                                                        pin_inputs=dict(
                                                        start_pin=dict(
                                                                component='LP1',
                                                                pin='tie'),
                                                       end_pin=dict(
                                                        component='CLT',
                                                        pin='prime_start')
                                                 )))


seg2_5 = RoutePathfinder(R5, 'seg2', options = dict(chip='main', 
                                                      trace_width =tl_width,
                                                      trace_gap =tl_gap,
                                                      fillet='100um',
                                                      hfss_wire_bonds = True,
                                                    lead=dict(start_straight = '0mm',
                                                        end_straight='0mm'),
                                                        pin_inputs=dict(
                                                        start_pin=dict(
                                                                component='CLT',
                                                                pin='prime_end'),
                                                       end_pin=dict(
                                                        component='LP2',
                                                        pin='tie')
                                                 )))


gui5.rebuild()
gui5.autoscale()


04:14PM 46s INFO [_start_renderers]: Renderer=gmsh skipped: runtime dependency not installed (renderer_gmsh requires gmsh. Install with: pip install 'quantum-metal[mesh]' (or the legacy alias 'quantum-metal[fem]')).


## Resonator 6: 6mm

In [8]:
#design and GUI intialization
R6 = designs.DesignPlanar()
R6.overwrite_enabled = True
gui6 = MetalGUI(R6)

#chip & sample holder dimensions
R6.chips.main.size.size_x = main_x
R6.chips.main.size.size_y = main_y
R6.chips.main.size.size_z = main_z
R6.chips.main.size.sample_holder_top = radbound_top
R6.chips.main.size.sample_holder_bottom = radbound_bottom

#launch pad definition
R6_LP1 = LaunchpadWirebondDriven(R6,name='LP1',
                       options={'orientation': '0', 'pos_x': '-2.1mm', 'pos_y': '0mm','pad_width':lp_width ,'pad_gap':lp_gap ,'pad_height':'180um', 'trace_width': tl_width , 'trace_gap': tl_gap},
                       component_template={'falseparam1': {'falseparam2': 'false-param', 'falseparam3': 'false-param'}},
                                        )

R6_LP2 = LaunchpadWirebondDriven(R6,name='LP2',
                       options={'orientation': '180', 'pos_x': '2.1mm', 'pos_y': '0mm','pad_width':lp_width ,'pad_gap':lp_gap, 'pad_height':'180um', 'trace_width':tl_width , 'trace_gap': tl_gap},
                       component_template={'falseparam1': {'falseparam2': 'false-param', 'falseparam3': 'false-param'}},
                                        )

#resonator definition
R6_OTG = OpenToGround(R6,name='OTG'
                 ,options={'pos_x':'-0.425mm','pos_y':'-1.0mm','orientation':'-90','width':rr_width, 'gap':rr_gap, 'termination_gap': rr_termination })


R6_CLT = CoupledLineTee(R6,'CLT', options=dict(pos_x = '0mm', pos_y = '0mm',
                                                        prime_width=tl_width,
                                                        prime_gap=tl_gap,
                                                        second_width=rr_width,
                                                        second_gap=rr_gap,
                                                        fillet='50um',
                                                        mirror=True,
                                                        orientation = '0',
                                                        coupling_space = rr_coupling_gap,                                                         
                                                        coupling_length = rr_coupling_length,
                                                        open_termination = False))

RR6 = RouteMeander(R6, 'RR6',  Dict(
        trace_width =tl_width,
        trace_gap =tl_gap,
        total_length='5150um',
        hfss_wire_bonds = True,
        fillet='50 um',
        lead = dict(start_straight='100um',end_straight='10um'),
        meander= dict(spacing='150um',asymmetry='200um'),
        pin_inputs=Dict(
            start_pin=Dict(component='CLT', pin='second_end'),
            end_pin=Dict(component='OTG', pin='open')), ))

#Tx line segments

seg1_6 = RoutePathfinder(R6, 'seg1', options = dict(chip='main', 
                                                      trace_width =tl_width,
                                                      trace_gap =tl_gap,
                                                      fillet='100um',
                                                      hfss_wire_bonds = True,
                                                    lead=dict(start_straight = '0mm',
                                                        end_straight='0mm'),
                                                        pin_inputs=dict(
                                                        start_pin=dict(
                                                                component='LP1',
                                                                pin='tie'),
                                                       end_pin=dict(
                                                        component='CLT',
                                                        pin='prime_start')
                                                 )))


seg2_6 = RoutePathfinder(R6, 'seg2', options = dict(chip='main', 
                                                      trace_width =tl_width,
                                                      trace_gap =tl_gap,
                                                      fillet='100um',
                                                      hfss_wire_bonds = True,
                                                    lead=dict(start_straight = '0mm',
                                                        end_straight='0mm'),
                                                        pin_inputs=dict(
                                                        start_pin=dict(
                                                                component='CLT',
                                                                pin='prime_end'),
                                                       end_pin=dict(
                                                        component='LP2',
                                                        pin='tie')
                                                 )))


gui6.rebuild()
gui6.autoscale()

04:14PM 47s INFO [_start_renderers]: Renderer=gmsh skipped: runtime dependency not installed (renderer_gmsh requires gmsh. Install with: pip install 'quantum-metal[mesh]' (or the legacy alias 'quantum-metal[fem]')).


## Resonator 7: 7mm


In [9]:
#design and GUI intialization
R7 = designs.DesignPlanar()
R7.overwrite_enabled = True
gui7 = MetalGUI(R7)

#chip & sample holder dimensions
R7.chips.main.size.size_x = main_x
R7.chips.main.size.size_y = main_y
R7.chips.main.size.size_z = main_z
R7.chips.main.size.sample_holder_top = radbound_top
R7.chips.main.size.sample_holder_bottom = radbound_bottom

#launch pad definition
R7_LP1 = LaunchpadWirebondDriven(R7,name='LP1',
                       options={'orientation': '0', 'pos_x': '-2.1mm', 'pos_y': '0mm','pad_width':lp_width ,'pad_gap':lp_gap ,'pad_height':'180um', 'trace_width': tl_width , 'trace_gap': tl_gap},
                       component_template={'falseparam1': {'falseparam2': 'false-param', 'falseparam3': 'false-param'}},
                                        )

R7_LP2 = LaunchpadWirebondDriven(R7,name='LP2',
                       options={'orientation': '180', 'pos_x': '2.1mm', 'pos_y': '0mm','pad_width':lp_width ,'pad_gap':lp_gap, 'pad_height':'180um', 'trace_width':tl_width , 'trace_gap': tl_gap},
                       component_template={'falseparam1': {'falseparam2': 'false-param', 'falseparam3': 'false-param'}},
                                        )

#resonator definition
R7_OTG = OpenToGround(R7,name='OTG'
                 ,options={'pos_x':'-0.425mm','pos_y':'-1.0mm','orientation':'-90','width':rr_width, 'gap':rr_gap, 'termination_gap': rr_termination })


R7_CLT = CoupledLineTee(R7,'CLT', options=dict(pos_x = '0mm', pos_y = '0mm',
                                                        prime_width=tl_width,
                                                        prime_gap=tl_gap,
                                                        second_width=rr_width,
                                                        second_gap=rr_gap,
                                                        fillet='50um',
                                                        mirror=True,
                                                        orientation = '0',
                                                        coupling_space = rr_coupling_gap,                                                         
                                                        coupling_length = rr_coupling_length,
                                                        open_termination = False))

RR7 = RouteMeander(R7, 'RR7',  Dict(
        trace_width =tl_width,
        trace_gap =tl_gap,
        total_length='6150um',
        hfss_wire_bonds = True,
        fillet='50 um',
        lead = dict(start_straight='100um',end_straight='10um'),
        meander= dict(spacing='150um',asymmetry='200um'),
        pin_inputs=Dict(
            start_pin=Dict(component='CLT', pin='second_end'),
            end_pin=Dict(component='OTG', pin='open')), ))

#Tx line segments

seg1_7 = RoutePathfinder(R7, 'seg1', options = dict(chip='main', 
                                                      trace_width =tl_width,
                                                      trace_gap =tl_gap,
                                                      fillet='100um',
                                                      hfss_wire_bonds = True,
                                                    lead=dict(start_straight = '0mm',
                                                        end_straight='0mm'),
                                                        pin_inputs=dict(
                                                        start_pin=dict(
                                                                component='LP1',
                                                                pin='tie'),
                                                       end_pin=dict(
                                                        component='CLT',
                                                        pin='prime_start')
                                                 )))


seg2_7 = RoutePathfinder(R7, 'seg2', options = dict(chip='main', 
                                                      trace_width =tl_width,
                                                      trace_gap =tl_gap,
                                                      fillet='100um',
                                                      hfss_wire_bonds = True,
                                                    lead=dict(start_straight = '0mm',
                                                        end_straight='0mm'),
                                                        pin_inputs=dict(
                                                        start_pin=dict(
                                                                component='CLT',
                                                                pin='prime_end'),
                                                       end_pin=dict(
                                                        component='LP2',
                                                        pin='tie')
                                                 )))


gui7.rebuild()
gui7.autoscale()


04:14PM 48s INFO [_start_renderers]: Renderer=gmsh skipped: runtime dependency not installed (renderer_gmsh requires gmsh. Install with: pip install 'quantum-metal[mesh]' (or the legacy alias 'quantum-metal[fem]')).


## Resonator 8: 8mm


In [10]:
#design and GUI intialization
R8 = designs.DesignPlanar()
R8.overwrite_enabled = True
gui8 = MetalGUI(R8)

#chip & sample holder dimensions
R8.chips.main.size.size_x = main_x
R8.chips.main.size.size_y = main_y
R8.chips.main.size.size_z = main_z
R8.chips.main.size.sample_holder_top = radbound_top
R8.chips.main.size.sample_holder_bottom = radbound_bottom

#launch pad definition
R8_LP1 = LaunchpadWirebondDriven(R8,name='LP1',
                       options={'orientation': '0', 'pos_x': '-2.1mm', 'pos_y': '0mm','pad_width':lp_width ,'pad_gap':lp_gap ,'pad_height':'180um', 'trace_width': tl_width , 'trace_gap': tl_gap},
                       component_template={'falseparam1': {'falseparam2': 'false-param', 'falseparam3': 'false-param'}},
                                        )

R8_LP2 = LaunchpadWirebondDriven(R8,name='LP2',
                       options={'orientation': '180', 'pos_x': '2.1mm', 'pos_y': '0mm','pad_width':lp_width ,'pad_gap':lp_gap, 'pad_height':'180um', 'trace_width':tl_width , 'trace_gap': tl_gap},
                       component_template={'falseparam1': {'falseparam2': 'false-param', 'falseparam3': 'false-param'}},
                                        )

#resonator definition
R8_OTG = OpenToGround(R8,name='OTG'
                 ,options={'pos_x':'-0.425mm','pos_y':'-1.0mm','orientation':'-90','width':rr_width, 'gap':rr_gap, 'termination_gap': rr_termination })


R8_CLT = CoupledLineTee(R8,'CLT', options=dict(pos_x = '0mm', pos_y = '0mm',
                                                        prime_width=tl_width,
                                                        prime_gap=tl_gap,
                                                        second_width=rr_width,
                                                        second_gap=rr_gap,
                                                        fillet='50um',
                                                        mirror=True,
                                                        orientation = '0',
                                                        coupling_space = rr_coupling_gap,                                                         
                                                        coupling_length = rr_coupling_length,
                                                        open_termination = False))

RR8 = RouteMeander(R8, 'RR8',  Dict(
        trace_width =tl_width,
        trace_gap =tl_gap,
        total_length='7150um',
        hfss_wire_bonds = True,
        fillet='50 um',
        lead = dict(start_straight='100um',end_straight='10um'),
        meander= dict(spacing='150um',asymmetry='200um'),
        pin_inputs=Dict(
            start_pin=Dict(component='CLT', pin='second_end'),
            end_pin=Dict(component='OTG', pin='open')), ))

#Tx line segments

seg1_8 = RoutePathfinder(R8, 'seg1', options = dict(chip='main', 
                                                      trace_width =tl_width,
                                                      trace_gap =tl_gap,
                                                      fillet='100um',
                                                      hfss_wire_bonds = True,
                                                    lead=dict(start_straight = '0mm',
                                                        end_straight='0mm'),
                                                        pin_inputs=dict(
                                                        start_pin=dict(
                                                                component='LP1',
                                                                pin='tie'),
                                                       end_pin=dict(
                                                        component='CLT',
                                                        pin='prime_start')
                                                 )))


seg2_8 = RoutePathfinder(R8, 'seg2', options = dict(chip='main', 
                                                      trace_width =tl_width,
                                                      trace_gap =tl_gap,
                                                      fillet='100um',
                                                      hfss_wire_bonds = True,
                                                    lead=dict(start_straight = '0mm',
                                                        end_straight='0mm'),
                                                        pin_inputs=dict(
                                                        start_pin=dict(
                                                                component='CLT',
                                                                pin='prime_end'),
                                                       end_pin=dict(
                                                        component='LP2',
                                                        pin='tie')
                                                 )))


gui8.rebuild()
gui8.autoscale()

04:14PM 49s INFO [_start_renderers]: Renderer=gmsh skipped: runtime dependency not installed (renderer_gmsh requires gmsh. Install with: pip install 'quantum-metal[mesh]' (or the legacy alias 'quantum-metal[fem]')).


## Resonator 9: 9mm


In [11]:
#design and GUI intialization
R9 = designs.DesignPlanar()
R9.overwrite_enabled = True
gui9 = MetalGUI(R9)

#chip & sample holder dimensions
R9.chips.main.size.size_x = main_x
R9.chips.main.size.size_y = main_y
R9.chips.main.size.size_z = main_z
R9.chips.main.size.sample_holder_top = radbound_top
R9.chips.main.size.sample_holder_bottom = radbound_bottom

#launch pad definition
R9_LP1 = LaunchpadWirebondDriven(R9,name='LP1',
                       options={'orientation': '0', 'pos_x': '-2.1mm', 'pos_y': '0mm','pad_width':lp_width ,'pad_gap':lp_gap ,'pad_height':'180um', 'trace_width': tl_width , 'trace_gap': tl_gap},
                       component_template={'falseparam1': {'falseparam2': 'false-param', 'falseparam3': 'false-param'}},
                                        )

R9_LP2 = LaunchpadWirebondDriven(R9,name='LP2',
                       options={'orientation': '180', 'pos_x': '2.1mm', 'pos_y': '0mm','pad_width':lp_width ,'pad_gap':lp_gap, 'pad_height':'180um', 'trace_width':tl_width , 'trace_gap': tl_gap},
                       component_template={'falseparam1': {'falseparam2': 'false-param', 'falseparam3': 'false-param'}},
                                        )

#resonator definition
R9_OTG = OpenToGround(R9,name='OTG'
                 ,options={'pos_x':'-0.425mm','pos_y':'-1.0mm','orientation':'-90','width':rr_width, 'gap':rr_gap, 'termination_gap': rr_termination })


R9_CLT = CoupledLineTee(R9,'CLT', options=dict(pos_x = '0mm', pos_y = '0mm',
                                                        prime_width=tl_width,
                                                        prime_gap=tl_gap,
                                                        second_width=rr_width,
                                                        second_gap=rr_gap,
                                                        fillet='50um',
                                                        mirror=True,
                                                        orientation = '0',
                                                        coupling_space = rr_coupling_gap,                                                         
                                                        coupling_length = rr_coupling_length,
                                                        open_termination = False))

RR9 = RouteMeander(R9, 'RR9',  Dict(
        trace_width =tl_width,
        trace_gap =tl_gap,
        total_length='8150um',
        hfss_wire_bonds = True,
        fillet='50 um',
        lead = dict(start_straight='100um',end_straight='10um'),
        meander= dict(spacing='150um',asymmetry='200um'),
        pin_inputs=Dict(
            start_pin=Dict(component='CLT', pin='second_end'),
            end_pin=Dict(component='OTG', pin='open')), ))

#Tx line segments

seg1_9 = RoutePathfinder(R9, 'seg1', options = dict(chip='main', 
                                                      trace_width =tl_width,
                                                      trace_gap =tl_gap,
                                                      fillet='100um',
                                                      hfss_wire_bonds = True,
                                                    lead=dict(start_straight = '0mm',
                                                        end_straight='0mm'),
                                                        pin_inputs=dict(
                                                        start_pin=dict(
                                                                component='LP1',
                                                                pin='tie'),
                                                       end_pin=dict(
                                                        component='CLT',
                                                        pin='prime_start')
                                                 )))


seg2_9 = RoutePathfinder(R9, 'seg2', options = dict(chip='main', 
                                                      trace_width =tl_width,
                                                      trace_gap =tl_gap,
                                                      fillet='100um',
                                                      hfss_wire_bonds = True,
                                                    lead=dict(start_straight = '0mm',
                                                        end_straight='0mm'),
                                                        pin_inputs=dict(
                                                        start_pin=dict(
                                                                component='CLT',
                                                                pin='prime_end'),
                                                       end_pin=dict(
                                                        component='LP2',
                                                        pin='tie')
                                                 )))


gui9.rebuild()
gui9.autoscale()

04:14PM 50s INFO [_start_renderers]: Renderer=gmsh skipped: runtime dependency not installed (renderer_gmsh requires gmsh. Install with: pip install 'quantum-metal[mesh]' (or the legacy alias 'quantum-metal[fem]')).


## Resonator 10: 10mm

In [12]:
#design and GUI intialization
R10 = designs.DesignPlanar()
R10.overwrite_enabled = True
gui10 = MetalGUI(R10)

#chip & sample holder dimensions
R10.chips.main.size.size_x = main_x
R10.chips.main.size.size_y = main_y
R10.chips.main.size.size_z = main_z
R10.chips.main.size.sample_holder_top = radbound_top
R10.chips.main.size.sample_holder_bottom = radbound_bottom

#launch pad definition
R10_LP1 = LaunchpadWirebondDriven(R10,name='LP1',
                       options={'orientation': '0', 'pos_x': '-2.1mm', 'pos_y': '0mm','pad_width':lp_width ,'pad_gap':lp_gap ,'pad_height':'180um', 'trace_width': tl_width , 'trace_gap': tl_gap},
                       component_template={'falseparam1': {'falseparam2': 'false-param', 'falseparam3': 'false-param'}},
                                        )

R10_LP2 = LaunchpadWirebondDriven(R10,name='LP2',
                       options={'orientation': '180', 'pos_x': '2.1mm', 'pos_y': '0mm','pad_width':lp_width ,'pad_gap':lp_gap, 'pad_height':'180um', 'trace_width':tl_width , 'trace_gap': tl_gap},
                       component_template={'falseparam1': {'falseparam2': 'false-param', 'falseparam3': 'false-param'}},
                                        )

#resonator definition
R10_OTG = OpenToGround(R10,name='OTG'
                 ,options={'pos_x':'-0.425mm','pos_y':'-1.0mm','orientation':'-90','width':rr_width, 'gap':rr_gap, 'termination_gap': rr_termination })


R10_CLT = CoupledLineTee(R10,'CLT', options=dict(pos_x = '0mm', pos_y = '0mm',
                                                        prime_width=tl_width,
                                                        prime_gap=tl_gap,
                                                        second_width=rr_width,
                                                        second_gap=rr_gap,
                                                        fillet='50um',
                                                        mirror=True,
                                                        orientation = '0',
                                                        coupling_space = rr_coupling_gap,                                                         
                                                        coupling_length = rr_coupling_length,
                                                        open_termination = False))

RR10 = RouteMeander(R10, 'RR10',  Dict(
        trace_width =tl_width,
        trace_gap =tl_gap,
        total_length='9150um',
        hfss_wire_bonds = True,
        fillet='50 um',
        lead = dict(start_straight='100um',end_straight='10um'),
        meander= dict(spacing='150um',asymmetry='200um'),
        pin_inputs=Dict(
            start_pin=Dict(component='CLT', pin='second_end'),
            end_pin=Dict(component='OTG', pin='open')), ))

#Tx line segments

seg1_10 = RoutePathfinder(R10, 'seg1', options = dict(chip='main', 
                                                      trace_width =tl_width,
                                                      trace_gap =tl_gap,
                                                      fillet='100um',
                                                      hfss_wire_bonds = True,
                                                    lead=dict(start_straight = '0mm',
                                                        end_straight='0mm'),
                                                        pin_inputs=dict(
                                                        start_pin=dict(
                                                                component='LP1',
                                                                pin='tie'),
                                                       end_pin=dict(
                                                        component='CLT',
                                                        pin='prime_start')
                                                 )))


seg2_10 = RoutePathfinder(R10, 'seg2', options = dict(chip='main', 
                                                      trace_width =tl_width,
                                                      trace_gap =tl_gap,
                                                      fillet='100um',
                                                      hfss_wire_bonds = True,
                                                    lead=dict(start_straight = '0mm',
                                                        end_straight='0mm'),
                                                        pin_inputs=dict(
                                                        start_pin=dict(
                                                                component='CLT',
                                                                pin='prime_end'),
                                                       end_pin=dict(
                                                        component='LP2',
                                                        pin='tie')
                                                 )))


gui10.rebuild()
gui10.autoscale()


04:14PM 51s INFO [_start_renderers]: Renderer=gmsh skipped: runtime dependency not installed (renderer_gmsh requires gmsh. Install with: pip install 'quantum-metal[mesh]' (or the legacy alias 'quantum-metal[fem]')).


## HFSS Export

In [13]:
# R1_pinfo = epr.ProjectInfo(
#     project_path=r'C:\Users\labuser\Documents\Ansoft\Thomas',
#     project_name='resonance_trend_investigation',
#     design_name='DM_1mm'
# )

# R1_hfss = R1.renderers.hfss
# R1_hfss.start()
# R1_hfss.activate_drivenmodal_design('DM_1mm')

# R1_hfss.render_design(
#     selection=[],                    # render all components
#     open_pins=[],
#     port_list=[('LP1', 'in', 50),    # lumped port at back of LP1 pad, 50 ohm
#                ('LP2', 'in', 50)],   # lumped port at back of LP2 pad, 50 ohm
# )



# R2_pinfo = epr.ProjectInfo(
#     project_path=r'C:\Users\labuser\Documents\Ansoft\Thomas',
#     project_name='resonance_trend_investigation',
#     design_name='DM_2mm'
# )

# R2_hfss = R2.renderers.hfss
# R2_hfss.start()
# R2_hfss.activate_drivenmodal_design('DM_2mm')

# R2_hfss.render_design(
#     selection=[],                    # render all components
#     open_pins=[],
#     port_list=[('LP1', 'in', 50),    # lumped port at back of LP1 pad, 50 ohm
#                ('LP2', 'in', 50)],   # lumped port at back of LP2 pad, 50 ohm
# )



# R3_pinfo = epr.ProjectInfo(
#     project_path=r'C:\Users\labuser\Documents\Ansoft\Thomas',
#     project_name='resonance_trend_investigation',
#     design_name='DM_3mm'
# )

# R3_hfss = R3.renderers.hfss
# R3_hfss.start()
# R3_hfss.activate_drivenmodal_design('DM_3mm')

# R3_hfss.render_design(
#     selection=[],                    # render all components
#     open_pins=[],
#     port_list=[('LP1', 'in', 50),    # lumped port at back of LP1 pad, 50 ohm
#                ('LP2', 'in', 50)],   # lumped port at back of LP2 pad, 50 ohm
# )



# R4_pinfo = epr.ProjectInfo(
#     project_path=r'C:\Users\labuser\Documents\Ansoft\Thomas',
#     project_name='resonance_trend_investigation',
#     design_name='DM_4mm'
# )

# R4_hfss = R4.renderers.hfss
# R4_hfss.start()
# R4_hfss.activate_drivenmodal_design('DM_4mm')

# R4_hfss.render_design(
#     selection=[],                    # render all components
#     open_pins=[],
#     port_list=[('LP1', 'in', 50),    # lumped port at back of LP1 pad, 50 ohm
#                ('LP2', 'in', 50)],   # lumped port at back of LP2 pad, 50 ohm
# )



# R5_pinfo = epr.ProjectInfo(
#     project_path=r'C:\Users\labuser\Documents\Ansoft\Thomas',
#     project_name='resonance_trend_investigation',
#     design_name='DM_5mm'
# )

# R5_hfss = R5.renderers.hfss
# R5_hfss.start()
# R5_hfss.activate_drivenmodal_design('DM_5mm')

# R5_hfss.render_design(
#     selection=[],                    # render all components
#     open_pins=[],
#     port_list=[('LP1', 'in', 50),    # lumped port at back of LP1 pad, 50 ohm
#                ('LP2', 'in', 50)],   # lumped port at back of LP2 pad, 50 ohm
# )



# R6_pinfo = epr.ProjectInfo(
#     project_path=r'C:\Users\labuser\Documents\Ansoft\Thomas',
#     project_name='resonance_trend_investigation',
#     design_name='DM_6mm'
# )

# R6_hfss = R6.renderers.hfss
# R6_hfss.start()
# R6_hfss.activate_drivenmodal_design('DM_6mm')

# R6_hfss.render_design(
#     selection=[],                    # render all components
#     open_pins=[],
#     port_list=[('LP1', 'in', 50),    # lumped port at back of LP1 pad, 50 ohm
#                ('LP2', 'in', 50)],   # lumped port at back of LP2 pad, 50 ohm
# )



# R7_pinfo = epr.ProjectInfo(
#     project_path=r'C:\Users\labuser\Documents\Ansoft\Thomas',
#     project_name='resonance_trend_investigation',
#     design_name='DM_7mm'
# )

# R7_hfss = R7.renderers.hfss
# R7_hfss.start()
# R7_hfss.activate_drivenmodal_design('DM_7mm')

# R7_hfss.render_design(
#     selection=[],                    # render all components
#     open_pins=[],
#     port_list=[('LP1', 'in', 50),    # lumped port at back of LP1 pad, 50 ohm
#                ('LP2', 'in', 50)],   # lumped port at back of LP2 pad, 50 ohm
# )



# R8_pinfo = epr.ProjectInfo(
#     project_path=r'C:\Users\labuser\Documents\Ansoft\Thomas',
#     project_name='resonance_trend_investigation',
#     design_name='DM_8mm'
# )

# R8_hfss = R8.renderers.hfss
# R8_hfss.start()
# R8_hfss.activate_drivenmodal_design('DM_8mm')

# R8_hfss.render_design(
#     selection=[],                    # render all components
#     open_pins=[],
#     port_list=[('LP1', 'in', 50),    # lumped port at back of LP1 pad, 50 ohm
#                ('LP2', 'in', 50)],   # lumped port at back of LP2 pad, 50 ohm
# )



# R9_pinfo = epr.ProjectInfo(
#     project_path=r'C:\Users\labuser\Documents\Ansoft\Thomas',
#     project_name='resonance_trend_investigation',
#     design_name='DM_9mm'
# )

# R9_hfss = R9.renderers.hfss
# R9_hfss.start()
# R9_hfss.activate_drivenmodal_design('DM_9mm')

# R9_hfss.render_design(
#     selection=[],                    # render all components
#     open_pins=[],
#     port_list=[('LP1', 'in', 50),    # lumped port at back of LP1 pad, 50 ohm
#                ('LP2', 'in', 50)],   # lumped port at back of LP2 pad, 50 ohm
# )



R10_pinfo = epr.ProjectInfo(
    project_path=r'C:\Users\labuser\Documents\Ansoft\Thomas',
    project_name='resonance_trend_investigation',
    design_name='iDM_10mm'
)

R10_hfss = R10.renderers.hfss
R10_hfss.start()
R10_hfss.activate_drivenmodal_design('iDM_10mm')

R10_hfss.render_design(
    selection=[],                    # render all components
    open_pins=[],
    port_list=[('LP1', 'in', 50),    # lumped port at back of LP1 pad, 50 ohm
               ('LP2', 'in', 50)],   # lumped port at back of LP2 pad, 50 ohm
)

INFO 04:14PM [connect_project]: Connecting to Ansys Desktop API...
INFO 04:14PM [load_ansys_project]: 	File path to HFSS project found.
INFO 04:14PM [load_ansys_project]: 	Opened Ansys App
INFO 04:14PM [load_ansys_project]: 	Opened Ansys Desktop v2025.1.0
INFO 04:14PM [load_ansys_project]: 	Opened Ansys Project
	Folder:    C:/Users/labuser/Documents/Ansoft/Thomas/
	Project:   resonance_trend_investigation
INFO 04:14PM [connect_design]: 	Opened active design
	Design:    iDM_10mm [Solution type: DrivenModal]
WARNING 04:14PM [connect_setup]: 	No design setup detected.
WARNING 04:14PM [connect_setup]: 	Creating driven modal default setup.
INFO 04:14PM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.HfssDMSetup'>)
INFO 04:14PM [connect]: 	Connected to project "resonance_trend_investigation" and design "iDM_10mm" 😀 

INFO 04:14PM [connect_project]: Connecting to Ansys Desktop API...
INFO 04:14PM [load_ansys_project]: 	Opened Ansys App
INFO 04:14PM [load_ansys_project]: 	Opened Ansys